# Stage 13 · Continue the Chair baselines and test the starting frequency mask

Attach the **original Chair kit** and **chair-s42-stage11-review.zip**, select **GPU T4 ×2** and **Run All**. The two Chair baselines resume their actual step-500 checkpoints. Each saves at 1,000, then continues to 2,000 if its foreground check passes. Save the **vanilla2000** archive before the diagnostic pair starts.

Two exploratory eight-view copies then restore the original regularised step-zero checkpoint and stop at **500**. The **first-band** copy starts with raw coordinates and the lowest sine/cosine band on both encoders and reveals the rest over the original ramp. The **near-only** control sees all bands immediately. Both retain the original near-camera weight of 0.01. The diagnostic copies record dead-field observations and continue to the short endpoint, so a white result can be inspected. The baselines keep the native early stop.

The measured rates imply about **4¼ minutes** of baseline optimisation plus **1¼–1½ minutes** for the diagnostic pair in parallel. Allow **8–15 minutes wall time** with loading, probes and saves. The per-worker remaining-training estimate refers to its full prefix. This notebook enforces the shorter limits above.

Send **chair-s42-stage13-review.zip** including when a diagnostic stays white. If execution raises, send the **diagnostics ZIP and traceback**. For a fresh session attach the newest Stage 13 ZIP alongside the two original inputs. Completed milestones are reused.

In [ ]:
from contextlib import ExitStack
from pathlib import Path
from tempfile import TemporaryDirectory
import base64
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
import torch
from IPython.display import HTML, display

inputs, outputs = Path("/kaggle/input"), Path("/kaggle/working")
work = outputs/"chair-stage13"
work.mkdir(exist_ok=True, parents=True)
if "stage13_temp" not in globals():
    stage13_temp = TemporaryDirectory(prefix="chair-probe-")
temp = Path(stage13_temp.name)
start_sha = "f88c4b4fcec9ffec8b34004eaa14d2e5939c266c28df6e2dca304cec9cfe582c"
review_sha = "80b250038e7335b82e982e4f576538f9128ef449a65650a3435f697a81040db5"
data_hash = "9c0224042d8d8060a2684ad70e6edfdf74f7b5acbd159aaa90142581c4e561e4"
if torch.cuda.device_count() < 2:
    raise RuntimeError("Select GPU T4 ×2 and restart this notebook.")
visible = os.environ.get("CUDA_VISIBLE_DEVICES")
devices = [n.strip() for n in visible.split(",")][:2] if visible else ["0", "1"]
assert len(set(devices)) == 2
print("Diagnostic workers:", [torch.cuda.get_device_name(i) for i in range(2)])

def read(path):
    return json.loads(path.read_bytes())

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def unpack(path):
    dest = temp/hashlib.sha256(str(path).encode()).hexdigest()
    dest.mkdir(exist_ok=True)
    with zipfile.ZipFile(path) as z:
        assert len(z.namelist()) == len(set(z.namelist()))
        for name in z.namelist():
            assert not Path(name).is_absolute() and ".." not in Path(name).parts
            z.extract(name, dest)
    return dest

def source(marker, filename, valid):
    for p in sorted(inputs.rglob(marker)):
        if valid(p.parent):
            return p.parent
    for p in sorted(inputs.rglob(filename)):
        d = unpack(p)
        if valid(d):
            return d
    raise FileNotFoundError(f"Attach the reviewed {filename} or its expanded contents.")

repo = source("RECOVERY_CHAIR.txt", "chair-input.zip", lambda d: read(d/"RECOVERY_CHAIR.txt")["data_hash"] == data_hash)
original = source("stage11-review.json", "chair-s42-stage11-review.zip", lambda d: sha(d/"stage11-review.json") == review_sha)
sys.path.insert(0, str(repo))
from recovery.data import verify_manifest
from recovery.health import dead_field
from recovery.media import digest, write_json
from recovery.training import fingerprint, implementation_hash, load_state, settings

review = read(original/"stage11-review.json")
assert implementation_hash() == review["implementation_hash"] and review["data_hash"] == data_hash
for name, expected in review["sha256"].items():
    assert (original/name).resolve().is_relative_to(original.resolve()) and sha(original/name) == expected
parent = original/"runs/chair/sparse-regularized/seed-42"
parent_meta = read(parent/"run.json")
initial, failure = read(parent/"initial-health.json"), read(parent/"health.json")
assert sha(parent/"checkpoint.pt") == start_sha and load_state(parent,parent_meta)["step"] == 0
assert failure["step"] == 128 and dead_field(failure) and initial["step"] == 0
manifest = work/"chair.json"
scene = verify_manifest(original/"manifests/chair.json", repo/"data/blender/chair")
assert scene["data_hash"] == data_hash and scene["splits"]["sparse"] == [26,86,2,55,75,93,16,73]
write_json(manifest,scene)
cfg = settings("main", scene["camera"])
assert cfg == parent_meta["training_config"] == review["matched_config"]["prefix_config"]
for arm in ("dense-vanilla","sparse-vanilla"):
    src = original/"runs/chair"/arm/"seed-42"
    meta = read(src/"run.json")
    health, init = read(src/"health.json"), read(src/"initial-health.json")
    assert (meta["protocol"], meta["training_config"], meta["seed"]) == (arm,cfg,42)
    assert load_state(src,meta)["step"]==health["step"]==500 and not dead_field(health)
    f=health["fine"]
    assert f["region_psnr"]>health["white_region_psnr"]+3 and f["region_opacity"]>.02 and f["region_mse"]<init["fine"]["region_mse"]
print("Verified both vanilla checkpoints at update 500 and the regularised initialisation at update zero.")

In [ ]:
worker_code = """
import json
import sys
from pathlib import Path
from unittest.mock import patch

import torch
from recovery import training
from recovery.health import dead_field, diagnostics, fixed_rays
from recovery.media import write_json
from sparse_nerf.data import SceneDataset
from sparse_nerf.model import FourierEncoding


def probe(model, dataset, rays, progress, coefficient):
    o,d,target,fg = rays
    captured = {}
    def capture(name):
        def save(module, args, values):
            captured[name] = values.detach()
        return save
    handles = [field.color.register_forward_hook(capture(name)) for name,field in (("coarse",model.coarse),("fine",model.fine))]
    try:
        out = model(o,d,dataset.near,dataset.far,regularized=True,progress=progress,
                    randomized=False,white_background=dataset.white_background)
    finally:
        for h in handles:
            h.remove()
    stats = {}
    for name,field,suffix in (("coarse",model.coarse,"_coarse"),("fine",model.fine,"")):
        rgb = out["rgb"+suffix][fg]
        mse = (rgb-target[fg]).square().mean()
        density,color = torch.autograd.grad(mse,(field.density.weight,field.color.weight),retain_graph=True)
        logits = captured[name][fg]
        assert all(t.isfinite().all() for t in (density,color,logits,rgb))
        stats[name] = dict(fg_rgb_min=rgb.min().item(), fg_rgb_mean=rgb.mean(0).tolist(),
            white_rgb_channel_fraction=(rgb==1).float().mean().item(),
            sample_sigmoid_exactly_one_fraction=(logits.sigmoid()==1).float().mean().item(),
            sample_logit_min=logits.min().item(), sample_logit_max=logits.max().item(),
            density_head_rgb_grad_l2_fp64=density.double().norm().item(),
            colour_head_rgb_grad_l2_fp64=color.double().norm().item(),
            colour_head_rgb_grad_nonzero=int(color.count_nonzero().item()))
        if name=="fine":
            near_grad, = torch.autograd.grad(out["near_loss"],field.density.weight)
            assert near_grad.isfinite().all()
            stats[name].update(near_loss=out["near_loss"].item(),weighted_near_loss=coefficient*out["near_loss"].item(),
                               near_density_head_grad_l2_unweighted=near_grad.double().norm().item(),
                               near_density_head_grad_l2_weighted=coefficient*near_grad.double().norm().item())
    return stats

def encoding_for(lane):
    native = FourierEncoding.forward
    if lane not in ("first-band", "near-only"):
        raise ValueError(f"Unknown encoding setting {lane}")

    def encode(module, xyz, fraction):
        if lane == "near-only" or fraction >= 1 or module.groups <= 3:
            return native(module, xyz, 1.)
        # The native mask exposes 1 + fraction * groups triplets. Interpolate
        # from raw XYZ + the first sin/cos pair to all triplets, separately for
        # the position and direction encoders.
        effective = (2 + fraction * (module.groups - 3)) / module.groups
        return native(module, xyz, effective)

    return encode


def run(folder, manifest, stop, device):
    meta = json.loads((folder/"run.json").read_bytes())
    cfg = meta["training_config"]
    args = (folder, manifest, meta["protocol"], meta["seed"], device, cfg)
    lane = meta.get("stage13_probe")
    if lane is None:
        assert meta["protocol"] in ("sparse-vanilla", "dense-vanilla") and stop <= 2000
        training.train(*args, resume=True, stop_at=stop, save_every=500)
        return
    assert 0 < stop <= 500 and cfg["stage13_encoding"] == lane and cfg["occ_weight"] == .01
    checks = []

    def observe(info):
        checks.append(dict(step=info["step"], native_dead=dead_field(info)))
        return False

    with patch.object(FourierEncoding, "forward", encoding_for(lane)):
        with patch.object(training, "dead_field", observe):
            training.train(*args, resume=True, stop_at=stop, save_every=500)
        state = training.load_state(folder, meta)
        model = training.make_model(cfg, device)
        model.load_state_dict(state["model"])
        dataset = SceneDataset(manifest, "sparse", cfg["downscale"])
        evidence = {}
        for name, count in (("original", 16), ("expanded", 32)):
            rays = fixed_rays(dataset, list(range(len(dataset))), device, count)
            progress = (stop-1) / (cfg["steps"]-1)
            health = diagnostics(model, dataset, rays, progress, True)
            evidence[name] = dict(health=health, measured=probe(model, dataset, rays, progress, cfg["occ_weight"]))
        write_json(folder/"probe.json", dict(lane=lane, step=stop, checks=checks, probes=evidence))


if __name__ == "__main__":
    run(Path(sys.argv[1]), Path(sys.argv[2]), int(sys.argv[3]), torch.device("cuda:0"))
"""
worker_path = temp/"chair_stage13.py"
worker_path.write_text(worker_code,encoding="utf8")

In [ ]:
vanillas = ("dense-vanilla", "sparse-vanilla")
candidates = ("first-band", "near-only")
vanilla_stops = (1000, 2000)
probe_stop = 500
experiment = digest(dict(worker=worker_code, stage11=review_sha, vanilla=vanilla_stops, probes=candidates, stop=probe_stop))
expected_rng = {"cpu_rng": "f55063e715227dd816a473455c671d22ea9212bf6d84e0c08e53669634bb7537",
                "cuda_rng": "173b960d9f274f38dffbd77dfb20dfd825f89bee5f557f4be701d1aece2e8977"}

def source_run(lane):
    arm = lane if lane in vanillas else "sparse-regularized"
    return original/"runs"/"chair"/arm/"seed-42"

def metadata(lane):
    base = read(source_run(lane)/"run.json")
    if lane in vanillas:
        return base
    return dict(base, run_id=digest([experiment,lane])[:32], stage13_probe=lane,
                parent_run_id=base["run_id"], parent_step0_sha256=start_sha,
                training_config=dict(base["training_config"], stage13_encoding=lane, stage13_experiment=experiment))

def location(lane, step):
    return Path("milestones")/lane/f"step-{step:06}"

def members(lane, step):
    files = ["run.json", "checkpoint.pt", "initial-health.json", "health.json", "training.jsonl",
             "console.log", f"train-{step:06}.png"]
    return files + ([] if lane in vanillas else ["probe.json"])

def foreground_pass(h, initial):
    f = h["fine"]
    return (not dead_field(h) and f["region_psnr"] > h["white_region_psnr"]+3
            and f["region_mse"] < initial["fine"]["region_mse"] and f["region_opacity"] > .02)

def check(run, lane, step):
    assert all((run/n).is_file() for n in members(lane,step)) and read(run/"run.json")==metadata(lane)
    state = load_state(run, metadata(lane))
    h, initial = read(run/"health.json"), read(run/"initial-health.json")
    log = [json.loads(row) for row in (run/"training.jsonl").read_bytes().splitlines()]
    assert state["step"] == h["step"] == log[-1]["step"] == step and initial["step"]==0
    assert [r["step"] for r in log] == sorted({r["step"] for r in log})
    slots = state["optimizer"]["state"]
    assert len(slots)==48 and all(v["step"].item()==step for v in slots.values())
    assert all(t.isfinite().all() for t in [*state["model"].values(), *(v[k] for v in slots.values() for k in ("exp_avg","exp_avg_sq"))])
    rng = {k: hashlib.sha256(state[k].numpy().tobytes()).hexdigest() for k in expected_rng}
    passed = foreground_pass(h,initial)
    if lane in candidates:
        p = read(run/"probe.json")
        assert p["lane"]==lane and p["step"]==step and rng==expected_rng
        assert dict(p["probes"]["original"]["health"],step=step)==h
    return dict(step=step,passed=passed,fg_psnr=h["fine"]["region_psnr"],white_psnr=h["white_region_psnr"],
                fg_opacity=h["fine"]["region_opacity"],segment_steps_per_s=log[-1]["steps_per_s"],
                checkpoint_sha256=sha(run/"checkpoint.pt"),rng_sha256=rng)

def persist(src, lane, step):
    check(src,lane,step)
    dest = work/location(lane,step)
    if dest.exists():
        assert check(dest,lane,step)==check(src,lane,step)
        return
    dest.parent.mkdir(parents=True,exist_ok=True)
    with TemporaryDirectory(dir=dest.parent) as temp_saved:
        stage = Path(temp_saved)/"ready"
        stage.mkdir()
        for name in members(lane,step):
            shutil.copyfile(src/name,stage/name)
        stage.rename(dest)

def latest(lane):
    stops = vanilla_stops if lane in vanillas else (probe_stop,)
    return next((s for s in reversed(stops) if (work/location(lane,s)).is_dir()), None)

def restore(folder):
    receipt = read(folder/"stage13-review.json")
    assert receipt["experiment"]==experiment and receipt["source_review_sha256"]==review_sha
    for name, value in receipt["sha256"].items():
        path = (folder/name).resolve()
        assert path.is_relative_to(folder.resolve()) and sha(path)==value
    for lane, record in receipt["records"].items():
        assert lane in vanillas+candidates
        step = record["step"]
        assert step in (vanilla_stops if lane in vanillas else (probe_stop,))
        rel = location(lane,step)
        assert all((rel/n).as_posix() in receipt["sha256"] for n in members(lane,step))
        assert check(folder/rel,lane,step)==record
        persist(folder/rel,lane,step)

for p in sorted(inputs.rglob("stage13-review.json")):
    restore(p.parent)
for p in sorted(inputs.rglob("chair-s42-stage13-*.zip")):
    restore(unpack(p))

def prepare(run,lane):
    last = latest(lane)
    src = work/location(lane,last) if last is not None else source_run(lane)
    if lane in vanillas or last is not None:
        for n in ("run.json","checkpoint.pt","training.jsonl","initial-health.json"):
            shutil.copyfile(src/n,run/n)
        return
    # New fingerprint and directory, original model, optimizer and random states.
    new_meta = metadata(lane)
    state = load_state(src,parent_meta)
    assert state["step"] == 0
    state["fingerprint"] = fingerprint(new_meta)
    torch.save(state,run/"checkpoint.pt")
    write_json(run/"run.json",new_meta)
    # train() records a fresh update-zero probe under this encoding when it resumes.

def snapshot(tag, errors, attempts=()):
    records, files = {}, {"manifests/chair.json":manifest}
    for lane in vanillas+candidates:
        step = latest(lane)
        if step is None:
            continue
        rel = location(lane,step)
        records[lane] = check(work/rel,lane,step)
        files.update({(rel/n).as_posix():work/rel/n for n in members(lane,step)})
    for lane, path in attempts:
        files.update({f"interrupted/{lane}/{p.name}":p for p in path.iterdir()
                      if p.suffix in (".json",".jsonl",".png",".pt",".log")})
    receipt = dict(tag=tag,experiment=experiment,source_review_sha256=review_sha,data_hash=data_hash,
                   implementation_hash=implementation_hash(),schedule_hash=review["schedule_hash"],
                   device_names=[torch.cuda.get_device_name(i) for i in range(2)],torch_version=str(torch.__version__),
                   records=records,errors=errors,sha256={k:sha(v) for k,v in files.items()})
    target = outputs/f"chair-s42-stage13-{tag}.zip"
    tmp_zip = target.with_suffix(".tmp")
    with zipfile.ZipFile(tmp_zip,"w",zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        z.writestr("stage13-review.json",json.dumps(receipt,indent=2,allow_nan=False))
        for name,path in files.items():
            z.write(path,name)
    with zipfile.ZipFile(tmp_zip) as z:
        assert z.testzip() is None and set(z.namelist())==set(files)|{"stage13-review.json"}
    tmp_zip.replace(target)
    print("Saved",target,flush=True)
    encoded = base64.b64encode(target.read_bytes()).decode("ascii")
    display(HTML(f'<a download="{target.name}" href="data:application/zip;base64,{encoded}">Download {target.name}</a>'))

def wave(pair, stop):
    errors, processes, attempts = {}, {}, {}
    with ExitStack() as stack:
        try:
            for i,lane in enumerate(pair):
                last = latest(lane)
                if last is not None:
                    status = check(work/location(lane,last),lane,last)
                    if last>=stop:
                        print(f"Reused {lane} at {last}.")
                        continue
                    if not status["passed"]:
                        print(f"Stopped {lane} after its foreground check at {last}.")
                        continue
                folder = Path(stack.enter_context(TemporaryDirectory(dir=work)))
                attempts[lane]=folder
                prepare(folder,lane)
                env = dict(os.environ,CUDA_VISIBLE_DEVICES=devices[i],OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",
                           PYTHONPATH=os.pathsep.join([str(repo),os.environ.get("PYTHONPATH","")]))
                log = stack.enter_context((folder/"console.log").open("wb"))
                reader = stack.enter_context((folder/"console.log").open(encoding="utf8",errors="replace"))
                cmd = [sys.executable,"-u",str(worker_path),str(folder),str(manifest),str(stop)]
                processes[lane]=(subprocess.Popen(cmd,cwd=repo,env=env,stdout=log,stderr=subprocess.STDOUT),reader)
            while processes:
                for lane,(proc,reader) in list(processes.items()):
                    code = proc.poll()
                    for line in reader:
                        print(f"[{lane}] {line.rstrip()}",flush=True)
                    if code is None:
                        continue
                    if code:
                        errors[lane]=f"Worker exited with code {code}."
                    else:
                        persist(attempts[lane],lane,stop)
                        info = check(work/location(lane,stop),lane,stop)
                        print(f"{lane} at {stop}: {info['fg_psnr']:.3f} dB, opacity {info['fg_opacity']:.4f}, foreground pass {info['passed']}.")
                    del processes[lane]
                if processes:
                    time.sleep(2)
        except BaseException:
            for proc,_ in processes.values():
                if proc.poll() is None:proc.terminate()
            for proc,_ in processes.values():proc.wait()
            snapshot("diagnostics",dict(errors,stage="Interrupted wave"),attempts.items())
            raise
        if errors:
            snapshot("diagnostics",errors,attempts.items())
            raise RuntimeError(f"Save the diagnostics ZIP and traceback: {errors}")

In [ ]:
for stop in vanilla_stops:
    wave(vanillas,stop)
    snapshot(f"vanilla{stop}",{})

In [ ]:
wave(candidates,probe_stop)
snapshot("review",{})
print("Stage 13 complete. Send chair-s42-stage13-review.zip for review.")